# Nanochat Distill Run on Kaggle

This notebook runs a Kaggle-friendly distillation flow for the `d8_kaggle` chat model:

- Uses `meta-llama/Llama-3.1-8B-Instruct` as teacher.
- Imports the saved SFT checkpoint cache from `nanochat-d8-sft-cache`.
- Generates teacher data in restartable JSONL shards.
- Keeps Hugging Face downloads in `/kaggle/temp` to protect `/kaggle/working` output space.
- Trains a distilled student from `sft:d8_kaggle` and writes `chatdistill_checkpoints/d8_kaggle`.

Recommended Kaggle setup: GPU on, Internet on, attach `nanochat-codes` and `nanochat-d8-sft-cache`.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

INPUT_ROOT = Path("/kaggle/input")

CODE_INPUTS = sorted(INPUT_ROOT.glob("**/nanochat-codes"))
SFT_CACHE_INPUTS = sorted(INPUT_ROOT.glob("**/nanochat-d8-sft-cache"))

assert CODE_INPUTS, "Attach the nanochat-codes Kaggle dataset"
assert SFT_CACHE_INPUTS, "Attach the nanochat-d8-sft-cache Kaggle dataset"

CODE_INPUT = CODE_INPUTS[0]
SFT_CACHE_INPUT = SFT_CACHE_INPUTS[0]

WORK_REPO = Path("/kaggle/working/nanochat")
WORK_CACHE = Path("/kaggle/working/nanochat_cache")
HF_CACHE = Path("/kaggle/temp/huggingface_cache")

for path in [WORK_REPO, WORK_CACHE, HF_CACHE]:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

shutil.copytree(CODE_INPUT, WORK_REPO, dirs_exist_ok=True)

# Distillation from sft:d8_kaggle needs the tokenizer and SFT checkpoint.
required_cache_paths = [
    Path("tokenizer"),
    Path("chatsft_checkpoints") / "d8_kaggle",
]
for rel_path in required_cache_paths:
    src = SFT_CACHE_INPUT / rel_path
    dst = WORK_CACHE / rel_path
    assert src.exists(), f"Missing required cache path in attached dataset: {src}"
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)

print("Code input:", CODE_INPUT)
print("SFT cache input:", SFT_CACHE_INPUT)
print("Working repo:", WORK_REPO)
print("Working cache:", WORK_CACHE)
print("HF cache:", HF_CACHE)
subprocess.run(["df", "-h", "/kaggle/working", "/kaggle/temp"], check=False)
subprocess.run(["du", "-sh", str(WORK_CACHE)], check=False)


## Install Dependencies

This run needs `bitsandbytes` for 8-bit Llama loading. The `only-if-needed` strategy avoids unnecessary upgrades of Kaggle's preinstalled CUDA/RAPIDS packages.

In [ ]:
packages = [
    "accelerate>=1.0.0",
    "bitsandbytes>=0.46.1",
    "datasets>=4.0.0",
    "filelock>=3.0.0",
    "psutil>=7.1.0",
    "requests>=2.32.0",
    "safetensors>=0.4.0",
    "sentencepiece>=0.2.0",
    "tiktoken>=0.11.0",
    "tokenizers>=0.22.0",
    "transformers>=4.57.3",
    "wandb>=0.21.3",
    "zstandard>=0.25.0",
    "rustbpe>=0.1.0",
]

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--upgrade-strategy",
    "only-if-needed",
] + packages)
print("Dependencies installed")


## Configure Runtime

In [ ]:
env = os.environ.copy()
env["NANOCHAT_BASE_DIR"] = str(WORK_CACHE)
env["HF_HOME"] = str(HF_CACHE)
env["HF_HUB_CACHE"] = str(HF_CACHE / "hub")
env["HF_DATASETS_CACHE"] = str(HF_CACHE / "datasets")
env["HF_HUB_DISABLE_XET"] = "1"
env["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"
env["HF_HUB_ETAG_TIMEOUT"] = "60"
env["TMPDIR"] = "/kaggle/temp"
env["PYTHONUNBUFFERED"] = "1"

# chat_distill.py does not use GradScaler, so fp32 is safer on T4 for student training.
env["NANOCHAT_DTYPE"] = "float32"

env["NANOCHAT_DISABLE_COMPILE"] = "1"
env["TORCH_COMPILE_DISABLE"] = "1"
env["NANOCHAT_ADAMW_ALLREDUCE"] = "1"
env["NANOCHAT_SERIAL_OPTIM_COMM"] = "1"
env["OMP_NUM_THREADS"] = "1"
env["NCCL_P2P_DISABLE"] = "1"
env["NCCL_IB_DISABLE"] = "1"
env["TORCH_NCCL_ASYNC_ERROR_HANDLING"] = "1"
env["NCCL_DEBUG"] = "WARN"

os.environ.update(env)

print("NANOCHAT_BASE_DIR:", env["NANOCHAT_BASE_DIR"])
print("HF_HOME:", env["HF_HOME"])
print("HF_HUB_DISABLE_XET:", env["HF_HUB_DISABLE_XET"])
print("NANOCHAT_DTYPE:", env["NANOCHAT_DTYPE"])


In [ ]:
import torch

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))


## Hugging Face Token

In [ ]:
import getpass

if "HF_TOKEN" not in os.environ or not os.environ["HF_TOKEN"]:
    os.environ["HF_TOKEN"] = getpass.getpass("HF_TOKEN: ")
env["HF_TOKEN"] = os.environ["HF_TOKEN"]
print("HF_TOKEN configured")


## Validate Repo And Cache

In [ ]:
assert (WORK_REPO / "scripts" / "chat_distill_data.py").exists(), "Missing scripts/chat_distill_data.py"
assert (WORK_REPO / "scripts" / "chat_distill.py").exists(), "Missing scripts/chat_distill.py"
assert (WORK_REPO / "nanochat" / "checkpoint_manager.py").exists(), "Missing nanochat/checkpoint_manager.py"

sft_dir = WORK_CACHE / "chatsft_checkpoints" / "d8_kaggle"
tokenizer_dir = WORK_CACHE / "tokenizer"
assert sft_dir.exists(), f"Missing SFT checkpoint dir: {sft_dir}"
assert tokenizer_dir.exists(), f"Missing tokenizer dir: {tokenizer_dir}"

subprocess.check_call(
    [
        sys.executable, "-m", "py_compile",
        "scripts/chat_distill_data.py",
        "scripts/chat_distill.py",
        "nanochat/checkpoint_manager.py",
    ],
    cwd=WORK_REPO,
    env=env,
)

print("Setup complete")
print("SFT checkpoints:", sorted(p.name for p in sft_dir.glob("model_*.pt"))[-5:])
print("Tokenizer files:", sorted(p.name for p in tokenizer_dir.iterdir()))


## Distill Run Settings

Defaults here are intentionally small enough for a Kaggle pass. If teacher generation is too slow, reduce `TOTAL_EXAMPLES` or `CHUNK_SIZE`.


In [ ]:
TEACHER_MODEL = "meta-llama/Llama-3.1-8B-Instruct"

TOTAL_EXAMPLES = 128
CHUNK_SIZE = 32
START_EXAMPLE = 0
MAX_NEW_TOKENS = 128
TEMPERATURE = 0.0

DISTILL_FORMAT = "sft"
DISTILL_DATA = WORK_CACHE / "teacher_sft_distill.jsonl"
SHARD_DIR = WORK_CACHE / "teacher_sft_distill_shards"
SHARD_DIR.mkdir(parents=True, exist_ok=True)

SYSTEM_PROMPT = (
    "You are a precise math tutor. Solve carefully and concisely. "
    "Always end with exactly one final line formatted as: #### <answer>"
)

print("Teacher:", TEACHER_MODEL)
print("Examples:", TOTAL_EXAMPLES)
print("Chunk size:", CHUNK_SIZE)
print("Max new tokens:", MAX_NEW_TOKENS)
print("Output:", DISTILL_DATA)


## Generate Teacher SFT Shards

Each shard is an independent JSONL file. If the notebook disconnects, rerun this cell and completed shards will be skipped.

In [ ]:
def count_jsonl(path):
    if not path.exists():
        return 0
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

for start in range(START_EXAMPLE, START_EXAMPLE + TOTAL_EXAMPLES, CHUNK_SIZE):
    end = min(START_EXAMPLE + TOTAL_EXAMPLES, start + CHUNK_SIZE)
    expected = end - start
    shard_path = SHARD_DIR / f"teacher_sft_{start:06d}_{end:06d}.jsonl"
    existing = count_jsonl(shard_path)
    if existing == expected:
        print(f"Skipping complete shard {shard_path.name} ({existing} rows)")
        continue
    if shard_path.exists():
        print(f"Regenerating incomplete shard {shard_path.name}: {existing}/{expected} rows")

    cmd = [
        sys.executable,
        "-m", "scripts.chat_distill_data",
        "--teacher-model", TEACHER_MODEL,
        "--input-source", "gsm8k",
        "--split", "train",
        "--start", str(start),
        "--max-examples", str(expected),
        "--output-format", DISTILL_FORMAT,
        "--max-new-tokens", str(MAX_NEW_TOKENS),
        "--temperature", str(TEMPERATURE),
        "--top-p", "0.95",
        "--torch-dtype", "float16",
        "--device-map", "auto",
        "--load-in-8bit", "1",
        "--chat-style", "solution",
        "--system-prompt", SYSTEM_PROMPT,
        "--output-path", str(shard_path),
    ]

    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)
    got = count_jsonl(shard_path)
    assert got == expected, f"Shard {shard_path} has {got} rows, expected {expected}"
    subprocess.run(["df", "-h", "/kaggle/working", "/kaggle/temp"], check=False)



## Merge And Inspect Teacher Data

In [ ]:
shard_paths = []
for start in range(START_EXAMPLE, START_EXAMPLE + TOTAL_EXAMPLES, CHUNK_SIZE):
    end = min(START_EXAMPLE + TOTAL_EXAMPLES, start + CHUNK_SIZE)
    shard_paths.append(SHARD_DIR / f"teacher_sft_{start:06d}_{end:06d}.jsonl")

rows = []
for path in shard_paths:
    assert path.exists(), f"Missing shard: {path}"
    with path.open("r", encoding="utf-8") as f:
        rows.extend(line.rstrip("\n") for line in f if line.strip())

DISTILL_DATA.write_text("\n".join(rows) + "\n", encoding="utf-8")

print("Merged data:", DISTILL_DATA)
print("Rows:", len(rows))
assert len(rows) == TOTAL_EXAMPLES, f"Expected {TOTAL_EXAMPLES} rows, got {len(rows)}"

parsed = [json.loads(line) for line in rows]
hash_count = sum("####" in example[-1]["content"] for example in parsed)
print("Rows containing #### in assistant answer:", hash_count, "/", len(parsed))
print(json.dumps(parsed[0], indent=2, ensure_ascii=False)[:3000])

In [ ]:
#Many teacher answers do not contain the expected #### pattern. 
#So the distill data does not teach the nanochat GSM8K answer format. 
#The model may learn some reasoning text, but the evaluator likely cannot extract the final answer.
#I’d fix data formatting before training.
#Easiest option: postprocess the teacher JSONL to 
#append a #### answer line when missing, using GSM8K’s gold answer from the source dataset.

work_repo_str = str(WORK_REPO)
if work_repo_str not in sys.path:
    sys.path.insert(0, work_repo_str)

from tasks.gsm8k import GSM8K, extract_answer

task = GSM8K(subset="main", split="train")

fixed_rows = []
fixed = 0

for i, line in enumerate(DISTILL_DATA.read_text(encoding="utf-8").splitlines()):
    row = json.loads(line)
    assistant = row[-1]["content"]

    if "####" not in assistant:
        source_idx = START_EXAMPLE + i
        answer_parts = task[source_idx]["messages"][-1]["content"]
        answer_text = "".join(part.get("text", "") for part in answer_parts if isinstance(part, dict))
        gold_answer = extract_answer(answer_text)
        assert gold_answer is not None, f"Could not extract gold answer for GSM8K source index {source_idx}"
        row[-1]["content"] = assistant.rstrip() + f"\n\n#### {gold_answer}"
        fixed += 1

    fixed_rows.append(json.dumps(row, ensure_ascii=False))

DISTILL_DATA.write_text("\n".join(fixed_rows) + "\n", encoding="utf-8")

print("Rows:", len(fixed_rows))
print("Fixed missing ####:", fixed)

## Optional Preference Data

This is off by default because the distill run already spends most of its time on teacher generation. Enable it only if you also want a preference dataset for later DPO/PPO/GRPO experiments.


In [ ]:
RUN_PREFERENCE_VARIANT = False
PREFERENCE_EXAMPLES = 128
PREFERENCE_PATH = WORK_CACHE / "teacher_prefs_distill.jsonl"

if RUN_PREFERENCE_VARIANT:
    cmd = [
        sys.executable,
        "-m", "scripts.chat_distill_data",
        "--teacher-model", TEACHER_MODEL,
        "--input-source", "gsm8k",
        "--split", "train",
        "--output-format", "preference",
        "--max-examples", str(PREFERENCE_EXAMPLES),
        "--max-new-tokens", str(MAX_NEW_TOKENS),
        "--temperature", str(TEMPERATURE),
        "--top-p", "0.95",
        "--torch-dtype", "float16",
        "--device-map", "auto",
        "--load-in-8bit", "1",
        "--chat-style", "solution",
        "--system-prompt", SYSTEM_PROMPT,
        "--rejected-style", "perturb",
        "--output-path", str(PREFERENCE_PATH),
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)
    print("Preference data:", PREFERENCE_PATH)
else:
    print("Skipping preference variant")


## Free Teacher Model Cache Before Student Training

After teacher data is generated, nanochat training no longer needs the Llama checkpoint. Freeing this space makes Kaggle output handling less fragile.

In [ ]:
FREE_TEACHER_CACHE_AFTER_DATA = True

if FREE_TEACHER_CACHE_AFTER_DATA:
    for path in [HF_CACHE / "hub" / "models--meta-llama--Llama-3.1-8B-Instruct", HF_CACHE / "xet"]:
        if path.exists():
            shutil.rmtree(path)
            print("Removed:", path)
else:
    print("Keeping teacher cache")

subprocess.run(["df", "-h", "/kaggle/working", "/kaggle/temp"], check=False)


## Train Distilled Student With Gentler LRs

In [ ]:
NPROC = 2 if torch.cuda.is_available() and torch.cuda.device_count() >= 2 else 1
STUDENT_SOURCE = "sft"
STUDENT_TAG = "d8_kaggle"

distill_args = [
    "-m", "scripts.chat_distill",
    "--",
    "--run", "dummy",
    "--student-source", STUDENT_SOURCE,
    "--student-tag", STUDENT_TAG,
    "--data-path", str(DISTILL_DATA),
    "--data-format", DISTILL_FORMAT,
    "--val-ratio", "0.1",
    "--num-epochs", "1",
    "--batch-size", "2",
    "--max-seq-len", "1024",
    "--embedding-lr", "0.01",
    "--matrix-lr", "0.001",
    "--unembedding-lr", "0.0002",
    "--warmup-ratio", "0.1",
    "--final-lr-frac", "0.1",
    "--eval-every", "25",
    "--save-every", "1000",
]

if NPROC > 1:
    cmd = ["torchrun", "--standalone", f"--nproc_per_node={NPROC}"] + distill_args
else:
    cmd = [sys.executable] + distill_args

print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)


## Inspect Distill Checkpoints

In [ ]:
distill_dir = WORK_CACHE / "chatdistill_checkpoints" / STUDENT_TAG
assert distill_dir.exists(), f"Missing distill checkpoint dir: {distill_dir}"

model_files = sorted(distill_dir.glob("model_*.pt"))
meta_files = sorted(distill_dir.glob("meta_*.json"))

print("Distill dir:", distill_dir)
print("Model checkpoints:", [p.name for p in model_files])
print("Metadata files:", [p.name for p in meta_files])

if meta_files:
    print(json.dumps(json.loads(meta_files[-1].read_text()), indent=2)[:3000])

subprocess.run(["du", "-sh", str(WORK_CACHE)], check=False)


## Comparison Eval

Use a larger limit than the smoke test if time allows. Scores are still noisy, but this is enough to catch obvious regressions.

In [ ]:
RUN_COMPARISON_EVAL = True
EVAL_MAX_PROBLEMS = 50

if RUN_COMPARISON_EVAL:
    post_eval_args = [
        "-m", "scripts.chat_post_eval",
        "--",
        "--models", f"sft=sft:{STUDENT_TAG}",
        "--models", f"distill=@{WORK_CACHE / 'chatdistill_checkpoints'}:{STUDENT_TAG}",
        "--max-problems", str(EVAL_MAX_PROBLEMS),
        "--batch-size", "2",
    ]
    if NPROC > 1:
        cmd = ["torchrun", "--standalone", f"--nproc_per_node={NPROC}"] + post_eval_args
    else:
        cmd = [sys.executable] + post_eval_args
    print("Running:", " ".join(str(x) for x in cmd))
    subprocess.run([str(x) for x in cmd], cwd=WORK_REPO, env=env, check=True)
else:
    print("Skipping comparison eval")


## Inspect Saved Comparison Report

In [ ]:
report_path = WORK_CACHE / "report" / "chat-post-eval.md"
print(report_path)
print("Exists:", report_path.exists())
if report_path.exists():
    print(report_path.read_text())


## Output Manifest

This records the main distill outputs in `/kaggle/working` for quick inspection.


In [ ]:
manifest = {
    "teacher_model": TEACHER_MODEL,
    "total_examples": TOTAL_EXAMPLES,
    "max_new_tokens": MAX_NEW_TOKENS,
    "distill_data": str(DISTILL_DATA),
    "preference_data": str(PREFERENCE_PATH) if PREFERENCE_PATH.exists() else None,
    "distill_checkpoint_dir": str(WORK_CACHE / "chatdistill_checkpoints" / STUDENT_TAG),
    "report": str(WORK_CACHE / "report" / "chat-post-eval.md"),
}
manifest_path = Path("/kaggle/working/nanochat_distill_manifest.json")
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(manifest_path)
print(manifest_path.read_text())

print("Main output files:")
for path in [DISTILL_DATA, PREFERENCE_PATH, manifest_path]:
    if path.exists():
        print(path, path.stat().st_size, "bytes")
for path in sorted((WORK_CACHE / "chatdistill_checkpoints" / STUDENT_TAG).glob("*")):
    print(path, path.stat().st_size, "bytes")


In [ ]:
# Optional: save the distill checkpoint cache as a Kaggle Dataset.
import json

DISTILL_MODEL_TAG = STUDENT_TAG
DISTILL_SOURCE_DIR = WORK_CACHE / 'chatdistill_checkpoints' / DISTILL_MODEL_TAG
TOKENIZER_SOURCE_DIR = WORK_CACHE / 'tokenizer'
DISTILL_CACHE_DIR = Path('/kaggle/working/nanochat_d8_distill_cache')

DATASET_ID = 'yshuaiqin/nanochat-d8-distill-cache'
TITLE = 'nanochat d8 distill cache'
VERSION_MESSAGE = f'Save {DISTILL_MODEL_TAG} chat distill checkpoint cache'
UPLOAD_ACTION = 'create'  # use 'version' after the Dataset already exists

assert '/' in DATASET_ID, "DATASET_ID should look like '<username>/<dataset-slug>'."
assert DISTILL_SOURCE_DIR.exists(), f'Missing distill checkpoint directory: {DISTILL_SOURCE_DIR}'
assert TOKENIZER_SOURCE_DIR.exists(), f'Missing tokenizer directory: {TOKENIZER_SOURCE_DIR}'
assert sorted(DISTILL_SOURCE_DIR.glob('model_*.pt')), f'No model_*.pt files found in {DISTILL_SOURCE_DIR}'
assert sorted(DISTILL_SOURCE_DIR.glob('meta_*.json')), f'No meta_*.json files found in {DISTILL_SOURCE_DIR}'

if DISTILL_CACHE_DIR.exists():
    shutil.rmtree(DISTILL_CACHE_DIR)
DISTILL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

shutil.copytree(WORK_CACHE / 'chatdistill_checkpoints', DISTILL_CACHE_DIR / 'chatdistill_checkpoints')
shutil.copytree(TOKENIZER_SOURCE_DIR, DISTILL_CACHE_DIR / 'tokenizer')

metadata = {
    'title': TITLE,
    'id': DATASET_ID,
    'licenses': [{'name': 'CC0-1.0'}],
}

metadata_path = DISTILL_CACHE_DIR / 'dataset-metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2))

print('Folder size:')
subprocess.run(['du', '-sh', str(DISTILL_CACHE_DIR)], check=False)

if UPLOAD_ACTION == 'create':
    cmd = [
        'kaggle', 'datasets', 'create',
        '-p', str(DISTILL_CACHE_DIR),
        '--dir-mode', 'tar',
    ]
elif UPLOAD_ACTION == 'version':
    cmd = [
        'kaggle', 'datasets', 'version',
        '-p', str(DISTILL_CACHE_DIR),
        '-m', VERSION_MESSAGE,
        '--dir-mode', 'tar',
    ]
else:
    raise ValueError("UPLOAD_ACTION must be 'create' or 'version'")

print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, text=True)

if result.returncode != 0:
    raise RuntimeError('Kaggle Dataset upload failed.')

print('Done.')
print(f'Dataset URL: https://www.kaggle.com/datasets/{DATASET_ID}')


## Serve the Distilled Chat Model

Run this after `chatdistill_checkpoints/d8_kaggle` exists. The first cell starts or restarts the local NanoChat web server from the distilled checkpoint. The second cell provides a small notebook chat loop against that server.


In [ ]:
import time
import requests

DISTILL_CHECKPOINT_DIR = WORK_CACHE / "chatdistill_checkpoints"
DISTILL_MODEL_TAG = STUDENT_TAG
SERVER_PORT = 8000
BASE_URL = f"http://127.0.0.1:{SERVER_PORT}"

model_dir = DISTILL_CHECKPOINT_DIR / DISTILL_MODEL_TAG
assert model_dir.exists(), f"Missing distill checkpoint directory: {model_dir}"
assert sorted(model_dir.glob("model_*.pt")), f"No model_*.pt files found in {model_dir}"
assert sorted(model_dir.glob("meta_*.json")), f"No meta_*.json files found in {model_dir}"

try:
    if server.poll() is None:
        print("Stopping existing NanoChat server...")
        server.terminate()
        server.wait(timeout=10)
        print("Stopped existing server.")
except NameError:
    pass
except Exception as exc:
    print("Could not stop existing server cleanly:", exc)
    try:
        server.kill()
        server.wait(timeout=10)
        print("Killed existing server.")
    except Exception:
        pass

messages = []

server_env = env.copy()
server_env["NANOCHAT_DISABLE_COMPILE"] = "1"
server_env["TORCH_COMPILE_DISABLE"] = "1"
server_env["OMP_NUM_THREADS"] = "1"

cmd = [
    sys.executable,
    "-m", "scripts.chat_web",
    "--checkpoint-dir", str(DISTILL_CHECKPOINT_DIR),
    "--model-tag", DISTILL_MODEL_TAG,
    "--num-gpus", "1",
    "--port", str(SERVER_PORT),
]

print("Starting NanoChat server:")
print(" ".join(cmd))
server = subprocess.Popen(cmd, cwd=WORK_REPO, env=server_env)
print(f"Server process started with PID {server.pid}.")

SERVER_READY = False
for _ in range(60):
    if server.poll() is not None:
        raise RuntimeError(f"NanoChat server exited early with code {server.returncode}")
    try:
        response = requests.get(f"{BASE_URL}/health", timeout=2)
        if response.ok:
            SERVER_READY = True
            break
    except requests.RequestException:
        pass
    time.sleep(2)

if SERVER_READY:
    print(f"NanoChat server is ready: {BASE_URL}")
else:
    print(f"NanoChat server is still loading. Wait a bit, then use: {BASE_URL}")


In [ ]:
import json
import requests

BASE = globals().get("BASE_URL", "http://127.0.0.1:8000")
messages = []

def ask(prompt, temperature=0.8, top_k=50, max_tokens=512):
    messages.append({"role": "user", "content": prompt})

    payload = {
        "messages": messages,
        "temperature": temperature,
        "top_k": top_k,
        "max_tokens": max_tokens,
    }

    print("Assistant:", end=" ", flush=True)
    answer = ""

    with requests.post(f"{BASE}/chat/completions", json=payload, stream=True, timeout=300) as response:
        response.raise_for_status()
        for line in response.iter_lines(decode_unicode=True):
            if not line or not line.startswith("data: "):
                continue

            data = json.loads(line[len("data: "):])
            if data.get("done"):
                break

            token = data.get("token", "")
            answer += token
            print(token, end="", flush=True)

    print()
    messages.append({"role": "assistant", "content": answer})
    return answer

print(f"Chatting with {BASE}. Type q, quit, or exit to stop. Type reset to clear history.")
while True:
    prompt = input("\nYou: ")
    command = prompt.strip().lower()
    if command in {"q", "quit", "exit"}:
        break
    if command == "reset":
        messages.clear()
        print("Chat history cleared.")
        continue
    ask(prompt)


In [ ]:
# Stop the NanoChat web server started by the serving cell.
import os
import time

SERVER_PORT = globals().get('SERVER_PORT', 8000)
stopped_any = False

server_process = globals().get('server')
if server_process is not None:
    if server_process.poll() is None:
        print(f'Stopping NanoChat server process {server_process.pid}...')
        server_process.terminate()
        try:
            server_process.wait(timeout=10)
            print('NanoChat server stopped.')
        except subprocess.TimeoutExpired:
            print('Server did not stop after terminate(); killing it...')
            server_process.kill()
            server_process.wait(timeout=10)
            print('NanoChat server killed.')
        stopped_any = True
    else:
        print(f'NanoChat server process already exited with code {server_process.returncode}.')
else:
    print('No notebook `server` variable found.')

try:
    import psutil

    current_pid = os.getpid()
    fallback_processes = []
    for proc in psutil.process_iter(['pid', 'name', 'cmdline']):
        if proc.info['pid'] == current_pid:
            continue
        cmdline = ' '.join(proc.info.get('cmdline') or [])
        if 'scripts.chat_web' not in cmdline:
            continue
        try:
            listening_on_port = any(
                conn.status == psutil.CONN_LISTEN
                and conn.laddr
                and conn.laddr.port == SERVER_PORT
                for conn in proc.net_connections(kind='inet')
            )
        except (psutil.AccessDenied, psutil.NoSuchProcess):
            listening_on_port = False
        if listening_on_port:
            fallback_processes.append(proc)

    for proc in fallback_processes:
        print(f'Stopping fallback chat_web process {proc.pid} on port {SERVER_PORT}...')
        proc.terminate()
    gone, alive = psutil.wait_procs(fallback_processes, timeout=10)
    for proc in alive:
        print(f'Killing fallback chat_web process {proc.pid}...')
        proc.kill()
    if fallback_processes:
        stopped_any = True
except Exception as exc:
    print('Fallback process scan skipped:', exc)

if stopped_any:
    time.sleep(2)
    print('Server shutdown complete.')
else:
    print('No running NanoChat server process found.')
